In [4]:
# ============================================================
# Geo-FNO training on nuclear steady-state xy slices
# - One sample = one (z level, BC temperature) pair
# - Input per point: [z, T, z^2, T^2, z * T]
# - Output per point: temperature on the corresponding xy slice
# - Uses Geo-FNO (FNO2d + IPHI) with the patched import path
# ============================================================

from __future__ import annotations

import json
import os
import sys
from dataclasses import dataclass
from pathlib import Path

import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.optim import Adam
from torch.utils.data import DataLoader, Dataset, Subset

THIS_DIR = Path.cwd()
PARENT_DIR = THIS_DIR.parent
if str(PARENT_DIR) not in sys.path:
    sys.path.insert(0, str(PARENT_DIR))

from geo_fno_def_patched import FNO2d, IPHI


@dataclass(frozen=True)
class NuclearSample:
    z_value: float
    temperature_value: float
    temperature_key: str


def relative_rmse(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    mse = torch.mean((pred - target) ** 2)
    ref = torch.mean(target ** 2)
    return torch.sqrt(mse / torch.clamp(ref, min=eps))


class NuclearSliceGeoFNODataset(Dataset):
    """
    One sample = one (z, boundary_temperature) slice.

    Input features per point:
      u_in[:, 0] = raw z level
      u_in[:, 1] = raw Dirichlet boundary temperature
      u_in[:, 2] = z^2
      u_in[:, 3] = T^2
      u_in[:, 4] = z * T

    Output per point:
      u_out[:, 0] = temperature at the corresponding (x, y) point
    """

    def __init__(
        self,
        h5_path: str,
        val_temperatures: list[float] | None = None,
        test_temperatures: list[float] | None = None,
        val_z_levels: list[float] | None = None,
        test_z_levels: list[float] | None = None,
    ):
        super().__init__()
        self.h5_path = h5_path
        self.f = h5py.File(h5_path, "r")
        self.coords = np.array(self.f["physical_field/coordinates"], dtype=np.float32)
        self.temp_group = self.f["physical_field/temperatures"]

        self.temperature_values = sorted(float(k.split("_")[1]) for k in self.temp_group.keys())
        self.z_levels = sorted(float(z) for z in np.unique(self.coords[:, 2]))

        self.z_mean = float(np.mean(self.z_levels))
        self.z_std = float(np.std(self.z_levels))
        self.temperature_mean = float(np.mean(self.temperature_values))
        self.temperature_std = float(np.std(self.temperature_values))

        self.xy_by_z: dict[float, np.ndarray] = {}
        self.mask_by_z: dict[float, np.ndarray] = {}
        self.center_by_z: dict[float, np.ndarray] = {}

        z_coords = self.coords[:, 2]
        for z_value in self.z_levels:
            mask = np.isclose(z_coords, z_value, atol=1e-8)
            xy = self.coords[mask, :2].astype(np.float32)
            self.mask_by_z[z_value] = mask
            self.xy_by_z[z_value] = xy
            self.center_by_z[z_value] = xy.mean(axis=0).astype(np.float32)

        self.samples = [
            NuclearSample(
                z_value=z_value,
                temperature_value=temp_value,
                temperature_key=f"SS_{int(temp_value)}",
            )
            for z_value in self.z_levels
            for temp_value in self.temperature_values
        ]

        self.val_temperatures = [float(v) for v in (val_temperatures or [300.0, 360.0, 420.0, 480.0, 540.0, 600.0, 660.0, 720.0, 780.0, 840.0, 900.0, 960.0])]
        self.test_temperatures = [float(v) for v in (test_temperatures or [330.0, 390.0, 450.0, 510.0, 570.0, 630.0, 690.0, 750.0, 810.0, 870.0, 930.0, 990.0])]
        self.requested_val_z_levels = [float(v) for v in (val_z_levels or [20.0, 320.0])]
        self.requested_test_z_levels = [float(v) for v in (test_z_levels or [30.0, 280.0])]
        self.val_z_levels = self._resolve_z_levels(self.requested_val_z_levels)
        self.test_z_levels = self._resolve_z_levels(self.requested_test_z_levels)

        self.xy_min = self.coords[:, :2].min(axis=0).astype(np.float32)
        self.xy_max = self.coords[:, :2].max(axis=0).astype(np.float32)

    def _resolve_z_levels(self, requested_levels: list[float]) -> list[float]:
        resolved = []
        for z_req in requested_levels:
            z_match = min(self.z_levels, key=lambda z: abs(z - float(z_req)))
            resolved.append(float(z_match))
        return resolved

    def __len__(self) -> int:
        return len(self.samples)

    def normalize_z(self, z_value: float) -> float:
        return (float(z_value) - self.z_mean) / max(self.z_std, 1e-6)

    def normalize_temperature(self, temperature_value: float) -> float:
        return (float(temperature_value) - self.temperature_mean) / max(self.temperature_std, 1e-6)

    def __getitem__(self, idx: int):
        sample = self.samples[idx]
        xy = self.xy_by_z[sample.z_value]
        mask = self.mask_by_z[sample.z_value]
        field = np.array(self.temp_group[sample.temperature_key], dtype=np.float32)[mask]

        z_value = float(sample.z_value)
        temperature_value = float(sample.temperature_value)

        u_in = np.empty((xy.shape[0], 5), dtype=np.float32)
        u_in[:, 0] = z_value
        u_in[:, 1] = temperature_value
        u_in[:, 2] = z_value * z_value
        u_in[:, 3] = temperature_value * temperature_value
        u_in[:, 4] = z_value * temperature_value

        u_out = field.reshape(-1, 1).astype(np.float32)

        code = np.zeros((42,), dtype=np.float32)
        code[0:2] = self.center_by_z[sample.z_value]

        return (
            torch.from_numpy(xy),
            torch.from_numpy(u_in),
            torch.from_numpy(u_out),
            torch.from_numpy(code),
        )

    def build_default_splits(self) -> tuple[list[int], list[int], list[int]]:
        val_pairs = {(z, t) for z in self.val_z_levels for t in self.val_temperatures}
        test_pairs = {(z, t) for z in self.test_z_levels for t in self.test_temperatures}

        train_idx: list[int] = []
        val_idx: list[int] = []
        test_idx: list[int] = []

        for idx, sample in enumerate(self.samples):
            pair = (sample.z_value, sample.temperature_value)
            if pair in test_pairs:
                test_idx.append(idx)
            elif pair in val_pairs:
                val_idx.append(idx)
            else:
                train_idx.append(idx)

        return train_idx, val_idx, test_idx


def compute_io_stats_from_indices(
    ds: NuclearSliceGeoFNODataset,
    indices: list[int],
    in_channels: int,
    out_channels: int,
) -> dict[str, torch.Tensor]:
    sum_in = torch.zeros(in_channels, dtype=torch.float64)
    sq_in = torch.zeros(in_channels, dtype=torch.float64)
    n_in = 0

    sum_out = torch.zeros(out_channels, dtype=torch.float64)
    sq_out = torch.zeros(out_channels, dtype=torch.float64)
    n_out = 0

    for i in indices:
        _, u_in, u_out, _ = ds[i]
        u_in64 = u_in.to(torch.float64)
        u_out64 = u_out.to(torch.float64)

        sum_in += u_in64.sum(dim=0)
        sq_in += (u_in64 * u_in64).sum(dim=0)
        n_in += int(u_in64.shape[0])

        sum_out += u_out64.sum(dim=0)
        sq_out += (u_out64 * u_out64).sum(dim=0)
        n_out += int(u_out64.shape[0])

    mean_in = sum_in / max(1, n_in)
    mean_out = sum_out / max(1, n_out)

    var_in = torch.clamp(sq_in / max(1, n_in) - mean_in * mean_in, min=1e-12)
    var_out = torch.clamp(sq_out / max(1, n_out) - mean_out * mean_out, min=1e-12)

    return {
        "u_in_mean": mean_in.to(torch.float32),
        "u_in_std": torch.sqrt(var_in).to(torch.float32),
        "u_out_mean": mean_out.to(torch.float32),
        "u_out_std": torch.sqrt(var_out).to(torch.float32),
    }


def evaluate(
    model: FNO2d,
    model_iphi: IPHI,
    loader: DataLoader,
    loss_fn: nn.Module,
    device: str,
) -> tuple[float, float]:
    model.eval()
    model_iphi.eval()
    total_mse = 0.0
    total_rel = 0.0

    with torch.no_grad():
        for pos, u_in, u_out, code in loader:
            pos = pos.to(device)
            u_in = u_in.to(device)
            u_out = u_out.to(device)
            code = code.to(device)

            pred = model(u_in, code=code, x_in=pos, x_out=pos, iphi=model_iphi)
            total_mse += float(loss_fn(pred, u_out).item())
            total_rel += float(relative_rmse(pred, u_out).item())

    total_mse /= max(1, len(loader))
    total_rel /= max(1, len(loader))
    return total_mse, total_rel


def save_geofno_checkpoint(
    model: FNO2d,
    model_iphi: IPHI,
    folder: str,
    name: str,
    extra_meta: dict | None = None,
):
    os.makedirs(folder, exist_ok=True)
    model_path = os.path.join(folder, f"{name}_fno.pt")
    iphi_path = os.path.join(folder, f"{name}_iphi.pt")
    meta_path = os.path.join(folder, f"{name}_meta.json")

    torch.save(model.state_dict(), model_path)
    torch.save(model_iphi.state_dict(), iphi_path)

    meta = {} if extra_meta is None else dict(extra_meta)
    meta["model_path"] = model_path
    meta["iphi_path"] = iphi_path
    with open(meta_path, "w") as f:
        json.dump(meta, f, indent=2)

    print("Saved:", model_path)
    print("Saved:", iphi_path)
    print("Saved:", meta_path)


def train_geofno_nuclear(
    h5_path: str = "/scratch/ROM_datasets_nico/Data_Single_Stack/simulation_data_complete.h5",
    batch_size: int = 8,
    epochs: int = 400,
    learning_rate_fno: float = 1e-3,
    learning_rate_iphi: float = 5e-5,
    weight_decay: float = 1e-6,
    modes: int = 16,
    width: int = 32,
    s1: int = 64,
    s2: int = 64,
    device: str | None = None,
    val_patience: int = 25,
    improve_thresh: float = 1e-3,
    use_io_normalization: bool = True,
):
    device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")
    ds = NuclearSliceGeoFNODataset(h5_path)
    train_idx, val_idx, test_idx = ds.build_default_splits()

    print(f"Loaded nuclear dataset with {len(ds)} samples")
    print("Unique z levels:", ds.z_levels)
    print("Temperature range:", ds.temperature_values[0], "to", ds.temperature_values[-1])
    print("Raw conditioning features: z, T, z^2, T^2, z*T")
    print("Train/val/test sizes:", len(train_idx), len(val_idx), len(test_idx))

    train_loader = DataLoader(Subset(ds, train_idx), batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(Subset(ds, val_idx), batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(Subset(ds, test_idx), batch_size=batch_size, shuffle=False)

    L1 = float(ds.xy_max[0] - ds.xy_min[0])
    L2 = float(ds.xy_max[1] - ds.xy_min[1])
    print("Using L:", [L1, L2])

    model = FNO2d(modes, modes, width, in_channels=5, out_channels=1, is_mesh=False, s1=s1, s2=s2, L=[L1, L2]).to(device)
    model_iphi = IPHI(width=32, device=device).to(device)

    norm_stats = None
    if use_io_normalization:
        norm_stats = compute_io_stats_from_indices(ds, train_idx, in_channels=5, out_channels=1)
        model.set_io_normalization(norm_stats["u_in_mean"], norm_stats["u_in_std"], norm_stats["u_out_mean"], norm_stats["u_out_std"], enabled=True)
        print("I/O normalization enabled")
        print("  u_in mean/std:", norm_stats["u_in_mean"].tolist(), norm_stats["u_in_std"].tolist())
        print("  u_out mean/std:", norm_stats["u_out_mean"].tolist(), norm_stats["u_out_std"].tolist())
    else:
        model.set_io_normalization_enabled(False)
        print("I/O normalization disabled")

    opt_fno = Adam(model.parameters(), lr=learning_rate_fno, weight_decay=weight_decay)
    opt_iphi = Adam(model_iphi.parameters(), lr=learning_rate_iphi, weight_decay=weight_decay)
    sched_fno = torch.optim.lr_scheduler.CosineAnnealingLR(opt_fno, T_max=epochs)
    sched_iphi = torch.optim.lr_scheduler.CosineAnnealingLR(opt_iphi, T_max=epochs)
    loss_fn = nn.MSELoss()

    best_val_rel = float("inf")
    best_epoch = -1
    epochs_no_improve = 0
    best_state_model = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    best_state_iphi = {k: v.detach().cpu().clone() for k, v in model_iphi.state_dict().items()}

    for ep in range(epochs):
        model.train()
        model_iphi.train()
        train_mse = 0.0
        train_rel = 0.0

        for pos, u_in, u_out, code in train_loader:
            pos = pos.to(device)
            u_in = u_in.to(device)
            u_out = u_out.to(device)
            code = code.to(device)

            opt_fno.zero_grad()
            opt_iphi.zero_grad()

            pred = model(u_in, code=code, x_in=pos, x_out=pos, iphi=model_iphi)
            mse = loss_fn(pred, u_out)
            mse.backward()

            opt_fno.step()
            opt_iphi.step()

            train_mse += float(mse.item())
            train_rel += float(relative_rmse(pred.detach(), u_out).item())

        train_mse /= max(1, len(train_loader))
        train_rel /= max(1, len(train_loader))
        val_mse, val_rel = evaluate(model, model_iphi, val_loader, loss_fn, device)

        sched_fno.step()
        sched_iphi.step()

        print(f"ep={ep:04d} train_relRMSE={train_rel:.6e} | val_relRMSE={val_rel:.6e} | train_MSE={train_mse:.6e} | val_MSE={val_mse:.6e}")

        if val_rel < best_val_rel * (1 - improve_thresh):
            best_val_rel = val_rel
            best_epoch = ep
            best_state_model = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            best_state_iphi = {k: v.detach().cpu().clone() for k, v in model_iphi.state_dict().items()}
            epochs_no_improve = 0
        else:
            if best_val_rel * (1 - improve_thresh) < val_rel < best_val_rel:
                best_val_rel = val_rel
                best_epoch = ep
                best_state_model = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_state_iphi = {k: v.detach().cpu().clone() for k, v in model_iphi.state_dict().items()}
            epochs_no_improve += 1
            if epochs_no_improve >= val_patience:
                print("Early stop triggered.")
                break

        if ep % 100 == 0:
            checkpoint_dir = THIS_DIR / "checkpoints"
            checkpoint_dir.mkdir(parents=True, exist_ok=True)
            torch.save(model.state_dict(), checkpoint_dir / f"geofno_nuclear_quadratic_ep{ep}.pt")
            torch.save(model_iphi.state_dict(), checkpoint_dir / f"iphi_nuclear_quadratic_ep{ep}.pt")

    model.load_state_dict(best_state_model)
    model_iphi.load_state_dict(best_state_iphi)

    test_mse, test_rel = evaluate(model, model_iphi, test_loader, loss_fn, device)
    print(f"Best val relRMSE: {best_val_rel:.6e} at epoch {best_epoch}")
    print(f"Test relRMSE: {test_rel:.6e} | Test MSE: {test_mse:.6e}")

    extra_meta = {
        "description": "Geo-FNO on nuclear steady-state slices conditioned on raw z, raw Dirichlet boundary temperature, quadratic terms, and interaction",
        "data_path": h5_path,
        "best_epoch": best_epoch,
        "val_metrics": {"mse": val_mse, "rel_l2": best_val_rel},
        "test_metrics": {"mse": test_mse, "rel_l2": test_rel},
        "split_config": {"val_temperatures": ds.val_temperatures, "test_temperatures": ds.test_temperatures, "val_z_levels": ds.val_z_levels, "test_z_levels": ds.test_z_levels},
        "feature_config": {"input_features": ["z", "T", "z^2", "T^2", "z*T"], "z_mean": ds.z_mean, "z_std": ds.z_std, "temperature_mean": ds.temperature_mean, "temperature_std": ds.temperature_std},
        "model_config": {"batch_size": batch_size, "epochs": epochs, "learning_rate_fno": learning_rate_fno, "learning_rate_iphi": learning_rate_iphi, "weight_decay": weight_decay, "modes1": modes, "modes2": modes, "width": width, "s1": s1, "s2": s2, "iphi_width": 32, "use_iphi": True},
        "norm_stats": null if norm_stats is None else {"cond_mean": norm_stats["u_in_mean"].tolist(), "cond_std": norm_stats["u_in_std"].tolist(), "target_mean": norm_stats["u_out_mean"].tolist(), "target_std": norm_stats["u_out_std"].tolist()},
        "xy_min": ds.xy_min.tolist(),
        "xy_max": ds.xy_max.tolist()
    }

    return model, model_iphi, extra_meta


In [5]:
h5_path = "/scratch/ROM_datasets_nico/Data_Single_Stack/simulation_data_complete.h5"
device = "cuda:0" if torch.cuda.is_available() else "cpu"

model, model_iphi, extra_meta = train_geofno_nuclear(
    h5_path=h5_path,
    batch_size=8,
    epochs=400,
    learning_rate_fno=1e-3,
    learning_rate_iphi=5e-5,
    weight_decay=1e-6,
    modes=16,
    width=32,
    s1=64,
    s2=64,
    device=device,
    val_patience=25,
    improve_thresh=1e-3,
    use_io_normalization=True,
)


Loaded nuclear dataset with 2820 samples
Unique z levels: [-50.0, -33.33333206176758, -16.66666603088379, -1.4210854715202004e-14, -7.105427357601002e-15, 0.0, 7.105427357601002e-15, 10.0, 20.0, 30.0, 40.0, 80.0, 160.0, 240.0, 280.0, 300.0, 320.0, 336.6666564941406, 353.3333435058594, 370.0]
Temperature range: 300.0 to 1000.0
Raw conditioning features: z, T, z^2, T^2, z*T
Train/val/test sizes: 2772 24 24
Using L: [36.0, 41.569217681884766]
I/O normalization enabled
  u_in mean/std: [121.29869842529297, 650.0866088867188, 36973.328125, 463999.21875, 78860.171875] [149.19769287109375, 203.43707275390625, 49227.23046875, 267082.96875, 104591.3125]
  u_out mean/std: [716.6375122070312] [215.9477996826172]
ep=0000 train_relRMSE=1.302835e-01 | val_relRMSE=3.158879e-02 | train_MSE=4.826853e+04 | val_MSE=5.241520e+02
ep=0001 train_relRMSE=4.368661e-02 | val_relRMSE=3.363496e-02 | train_MSE=1.133799e+03 | val_MSE=6.062910e+02
ep=0002 train_relRMSE=3.544658e-02 | val_relRMSE=4.156076e-02 | train

In [6]:
save_geofno_checkpoint(
    model,
    model_iphi,
    folder="/scratch/mnhagen/models/nuclear_dataset/old",
    name="geofno_htgr_z_T_z^2_T^2_z*T",
    extra_meta=extra_meta,
)


Saved: /scratch/mnhagen/models/nuclear_dataset/old/geofno_htgr_z_T_z^2_T^2_z*T_fno.pt
Saved: /scratch/mnhagen/models/nuclear_dataset/old/geofno_htgr_z_T_z^2_T^2_z*T_iphi.pt
Saved: /scratch/mnhagen/models/nuclear_dataset/old/geofno_htgr_z_T_z^2_T^2_z*T_meta.json
